In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ===============================
# 1️⃣ 读取预测结果
# ===============================
json_path = "/content/drive/MyDrive/Qwen3VL4B_HM_ZeroShot_pred.json"

with open(json_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)
pred_df["image_id"] = (
    pred_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .str.replace(".jpeg", "", regex=False)
    .str.replace(".png", "", regex=False)
    .str.replace(".gif", "", regex=False)
    .str.strip()
)

# ===============================
# 2️⃣ 读取真实标签
# ===============================
excel_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_labels.xlsx"
true_df = pd.read_excel(excel_path)
true_df.columns = true_df.columns.str.strip().str.lower()
true_df["image_id"] = true_df["id"].astype(str).str.strip()

# ===============================
# 3️⃣ 合并
# ===============================
merged_df = pd.merge(true_df, pred_df, on="image_id", how="inner")

print("Total True Labels:", len(true_df))
print("Total Predictions:", len(pred_df))
print("After Merge:", len(merged_df))

# ===============================
# 4️⃣ 标签标准化
# ===============================
def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["homophobia", "homophobic"]:
        return "homophobic"
    if x in ["transphobia", "transphobic"]:
        return "transphobic"
    if x in ["non_lgbt", "non-above", "non_anti_lgbt", "non anti lgbt"]:
        return "non anti lgbt"
    return x

merged_df["true_label"] = merged_df["label"].apply(normalize_label)
merged_df["predicted_label"] = merged_df["predicted_label"].apply(normalize_label)

print("\nUnique TRUE labels:", sorted(merged_df["true_label"].unique()))
print("Unique PRED labels:", sorted(merged_df["predicted_label"].unique()))

# ===============================
# 5️⃣ 计算指标
# ===============================
y_true = merged_df["true_label"]
y_pred = merged_df["predicted_label"]

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Total True Labels: 239
Total Predictions: 232
After Merge: 232

Unique TRUE labels: ['homophobic', 'non anti lgbt', 'transphobic']
Unique PRED labels: ['homophobic', 'non anti lgbt', 'transphobic']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.3664
MP   : 0.7028
MR   : 0.5354
MF1  : 0.4249
WP   : 0.8332
WR   : 0.3664
WF1  : 0.3389
-------------------------------------------------

Confusion Matrix:
[[ 30 138   1]
 [  0  49   0]
 [  0   8   6]]

Classification Report:
               precision    recall  f1-score   support

   homophobic       1.00      0.18      0.30       169
non anti lgbt       0.25      1.00      0.40        49
  transphobic       0.86      0.43      0.57        14

     accuracy                           0.37       232
    macro avg       0.70      0.54      0.42       232
 weighted avg       0.83      0.37      0.34       232

